# Dataset Builder: Bioinformatics & Drug Discovery

Notebook ini membangun **dataset *processed* (20 tabel relasional) dari tiga sumber publik, lalu mengekspornya ke **JSON** (`dataset/json/<tabel>.json`) dan **XLSX** (`dataset/dataset.xlsx`).

| Sumber | Dipakai untuk | Akses |
|---|---|---|
| **DrugCentral** | obat, target, bioaktivitas, asosiasi obat-penyakit, ATC, approval, sinyal efek samping FAERS | PostgreSQL publik **atau** salinan lokal (`.pgdump`) |
| **UniProt** | protein, organisme, family protein, anotasi Gene Ontology (GO) | REST API |
| **Reactome** | pathway *Homo sapiens* dan hierarkinya | berkas unduhan TSV |

**Alur:** Konfigurasi & Sumber Data -> Ekstrak (atau muat Snapshot / CSV lama) -> Skema -> Bersihkan & Normalisasi -> Validasi -> Ekspor (`dataset_export.zip`).


In [ ]:
!pip -q install psycopg2-binary requests xlsxwriter

In [ ]:
import hashlib
import io
import json
import re
import socket
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
import requests

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

# Konfigurasi direktori & provenance
OUT = Path("dataset")
OUT_JSON = OUT / "json"
OUT_JSON.mkdir(parents=True, exist_ok=True)

EXCEL_MAX = 1_048_576
RAW = {}
PROV = {"generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}

# Utilitas umum
def num(s):
    """Konversi aman ke tipe integer nullable pandas."""
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def keep(df, cols, name):
    """Ambil kolom yang diharapkan; beri peringatan jika skema sumber berubah."""
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print(f"[WARN] {name}: kolom tidak ditemukan -> {miss}")
    return df[[c for c in cols if c in df.columns]].copy()


def assign_ids(df, id_col, prefix, width=7):
    """ID sintetis berurutan yang deterministik: baris diurutkan menurut isinya
    dulu sehingga menjalankan ulang pada data yang sama menghasilkan ID sama."""
    key = pd.Series(
        ["\x1f".join(map(str, row)) for row in df.itertuples(index=False, name=None)],
        index=df.index,
    )
    df = (
        df.assign(_k=key)
        .sort_values("_k", kind="mergesort")
        .drop(columns="_k")
        .reset_index(drop=True)
    )
    df.insert(0, id_col, [f"{prefix}{i:0{width}d}" for i in range(len(df))])
    return df


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _py(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    raise TypeError(type(o))

## 1. Sumber Data Mentah

### 1.0 Pilih Mode Sumber Data & Koneksi DrugCentral
Pipeline memeriksa 3 jalur secara berurutan:
1. **`LEGACY_CSV_DIR`**: Memuat folder CSV hasil *run* lama (melewati ekstraksi 1A–1C).
2. **`SNAP` (`raw_cache/raw_snapshot.pkl`)**: Memuat *snapshot* hasil ekstraksi sebelumnya (melewati ekstraksi 1A–1C).
3. **Ekstraksi langsung** dari DrugCentral (PostgreSQL publik **atau** lokal dari `.pgdump`), UniProt REST API, dan Reactome.

Jika server publik DrugCentral (`unmtid-dbs.net:5433`) sedang tidak dapat dihubungi, ubah `USE_LOCAL_PG = True` lalu jalankan sel **1.0b** untuk me-*restore* file `.pgdump` ke PostgreSQL lokal di Colab.

In [ ]:
# --- Jalur sumber data ---
LEGACY_CSV_DIR = None
FORCE_REFRESH  = False

# --- Mode koneksi PostgreSQL DrugCentral ---
USE_LOCAL_PG = False

if USE_LOCAL_PG:
    DC_CONN = dict(
        host="localhost",
        port=5432,
        dbname="drugcentral",
        user="postgres",
        password="postgres",
        connect_timeout=30,
    )
else:
    DC_CONN = dict(
        host="unmtid-dbs.net",
        port=5433,
        dbname="drugcentral",
        user="drugman",
        password="dosage",
        connect_timeout=30,
    )

# --- Lokasi snapshot ---
SNAP = Path("raw_cache") / "raw_snapshot.pkl"
from google.colab import drive; drive.mount("/content/drive")
SNAP = Path("/content/drive/MyDrive/kgqa/raw_snapshot.pkl")
SNAP.parent.mkdir(parents=True, exist_ok=True)

RAW_KEYS = [
    "bioactivity", "target", "target_protein", "drug", "indication", "drug_atc",
    "atc_class", "approval", "adverse_signal", "protein", "organism",
    "protein_go", "go_term", "protein_family",
    "pathway", "pathway_hierarchy", "protein_pathway",
]

USE_SNAPSHOT = False

if LEGACY_CSV_DIR:
    d = Path(LEGACY_CSV_DIR)
    missing = [k for k in RAW_KEYS if not (d / f"{k}.csv").exists()]
    assert not missing, f"CSV lama tidak lengkap di {d}: {missing}"
    for k in RAW_KEYS:
        RAW[k] = pd.read_csv(d / f"{k}.csv")
    RAW["bioactivity"] = assign_ids(RAW["bioactivity"].drop(columns="act_id"), "act_id", "ACT")
    PROV["sumber_data_mentah"] = {
        "mode": "csv_lama",
        "folder": str(d),
        "catatan": "tanpa metadata versi sumber",
    }
    USE_SNAPSHOT = True
    print(f"Memakai CSV lama dari {d}: {len(RAW_KEYS)} tabel mentah dimuat. Ekstraksi 1A-1C dilewati.")
elif SNAP.exists() and not FORCE_REFRESH:
    snap = pd.read_pickle(SNAP)  # hanya muat berkas pickle yang Anda buat sendiri
    RAW.update(snap["RAW"])
    PROV.update({k: v for k, v in snap["PROV"].items() if k != "generated_at_utc"})
    PROV["sumber_data_mentah"] = {
        "mode": "snapshot",
        "berkas": str(SNAP),
        "dibuat_utc": snap["PROV"].get("generated_at_utc"),
    }
    USE_SNAPSHOT = True
    print(f"Memakai snapshot {SNAP} (dibuat {snap['PROV'].get('generated_at_utc')}). Ekstraksi 1A-1C dilewati.")
else:
    PROV["sumber_data_mentah"] = {
        "mode": "ekstraksi_langsung",
        "drugcentral_mode": "localhost_pg_restore" if USE_LOCAL_PG else "public_postgres",
    }
    print(f"Tidak ada snapshot/CSV lama: ekstraksi langsung akan dijalankan (DrugCentral: {DC_CONN['host']}:{DC_CONN['port']}).")

Mounted at /content/drive
Tidak ada snapshot/CSV lama: ekstraksi langsung akan dijalankan (DrugCentral: unmtid-dbs.net:5433).


### 1.0b (Opsional) Setup PostgreSQL Lokal & Restore `.pgdump` di Colab
Jalankan sel ini **hanya jika `USE_LOCAL_PG = True`** dan Anda memiliki berkas `.pgdump` DrugCentral (misalnya dari Google Drive). Jika `USE_LOCAL_PG = False` atau menggunakan snapshot, sel ini otomatis dilewati.

In [ ]:
PGDUMP_PATH = "/content/drive/MyDrive/drugcentral.pgdump"  # sesuaikan path file .pgdump Anda

if USE_LOCAL_PG and not USE_SNAPSHOT:
    import subprocess
    assert Path(PGDUMP_PATH).exists(), f"Berkas dump tidak ditemukan: {PGDUMP_PATH}"
    cmds = [
        "apt-get -qq update && apt-get -qq install -y postgresql",
        "service postgresql start",
        'sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD \'postgres\';"',
        "sudo -u postgres dropdb --if-exists drugcentral && sudo -u postgres createdb drugcentral",
        f'sudo -u postgres pg_restore -d drugcentral --no-owner --no-privileges "{PGDUMP_PATH}" || true',
    ]
    for cmd in cmds:
        print(">>>", cmd)
        subprocess.run(cmd, shell=True, check=True)
    print("Restore selesai. Memverifikasi 7 tabel utama DrugCentral...")
    verify_sql = (
        "SELECT 'act_table_full' AS tabel, count(*) FROM act_table_full UNION ALL "
        "SELECT 'structures', count(*) FROM structures UNION ALL "
        "SELECT 'omop_relationship', count(*) FROM omop_relationship UNION ALL "
        "SELECT 'atc', count(*) FROM atc UNION ALL "
        "SELECT 'struct2atc', count(*) FROM struct2atc UNION ALL "
        "SELECT 'approval', count(*) FROM approval UNION ALL "
        "SELECT 'faers', count(*) FROM faers;"
    )
    subprocess.run(f'sudo -u postgres psql -d drugcentral -c "{verify_sql}"', shell=True, check=True)
else:
    print("Melewati setup PostgreSQL lokal.")

Melewati setup PostgreSQL lokal.


### 1A. Ekstraksi DrugCentral
Mengambil 7 tabel (`act_table_full`, `structures`, `omop_relationship`, `atc`, `struct2atc`, `approval`, `faers`) dari PostgreSQL yang dikonfigurasi di `DC_CONN`.

In [ ]:
def check_connectivity():
    host, port = DC_CONN["host"], DC_CONN["port"]
    s = socket.socket()
    s.settimeout(10)
    ok = s.connect_ex((host, port)) == 0
    s.close()
    print(f"port {port} ({host}):", "TERBUKA" if ok else "DITOLAK/TERTUTUP")
    if not ok:
        raise RuntimeError(
            f"DrugCentral ({host}:{port}) tidak dapat dihubungi. "
            "Opsi: (1) coba lagi nanti, (2) pakai snapshot / LEGACY_CSV_DIR, "
            "atau (3) set USE_LOCAL_PG = True dan restore .pgdump di sel 1.0b."
        )


def connect_dc(attempts=6, wait=20):
    last = None
    for k in range(1, attempts + 1):
        try:
            return psycopg2.connect(**DC_CONN)
        except psycopg2.OperationalError as e:
            last = e
            print(f"[{k}/{attempts}] koneksi DrugCentral gagal: {str(e).splitlines()[0]}")
            if k < attempts:
                time.sleep(wait * k)
    raise RuntimeError(
        "DrugCentral tidak dapat dihubungi setelah beberapa percobaan."
    ) from last


def extract_drugcentral():
    check_connectivity()
    conn = connect_dc()
    q = lambda sql: pd.read_sql(sql, conn)

    # --- Bioaktivitas, target, target_protein ---
    act = q("SELECT * FROM act_table_full")
    ACT_COLS = [
        "struct_id", "target_id", "act_value", "act_unit", "act_type",
        "act_comment", "act_source", "relation", "moa", "action_type",
    ]
    RAW["bioactivity"] = assign_ids(keep(act, ACT_COLS, "bioactivity"), "act_id", "ACT")

    TGT_COLS = ["target_id", "target_name", "target_class", "organism", "tdl"]
    tgt_all = keep(act, TGT_COLS, "target").drop_duplicates()
    print("target dengan atribut tidak konsisten antar-baris:", int(tgt_all.target_id.duplicated().sum()))
    RAW["target"] = tgt_all.sort_values("target_id", kind="mergesort").drop_duplicates("target_id")

    tp = act[["target_id", "accession"]].dropna().drop_duplicates()
    tp = tp.assign(accession=tp.accession.str.split("|")).explode("accession")
    tp["accession"] = tp.accession.str.strip()
    RAW["target_protein"] = tp.drop_duplicates()

    # --- Obat ---
    RAW["drug"] = keep(
        q("SELECT * FROM structures"),
        ["id", "name", "cas_reg_no", "smiles", "inchikey", "cd_molweight", "cd_formula"],
        "drug",
    ).rename(columns={"id": "struct_id"})

    # --- Asosiasi obat-penyakit ---
    ind = q("SELECT * FROM omop_relationship")
    RAW["indication"] = keep(
        ind,
        ["struct_id", "concept_id", "concept_name", "relationship_name", "snomed_conceptid", "umls_cui"],
        "indication",
    )

    # --- ATC ---
    atc = q("SELECT * FROM atc")
    s2a = q("SELECT * FROM struct2atc")
    RAW["drug_atc"] = keep(s2a, ["struct_id", "atc_code"], "drug_atc")
    nodes = []
    for L in range(1, 5):
        c, n = f"l{L}_code", f"l{L}_name"
        if {c, n} <= set(atc.columns):
            t = atc[[c, n]].drop_duplicates()
            t.columns = ["atc_code", "atc_name"]
            t["level"] = L
            nodes.append(t)
    if {"code", "chemical_substance"} <= set(atc.columns):
        t = atc[["code", "chemical_substance"]].drop_duplicates()
        t.columns = ["atc_code", "atc_name"]
        t["level"] = 5
        nodes.append(t)
    atc_nodes = pd.concat(nodes).dropna().drop_duplicates("atc_code")
    plen = {2: 1, 3: 3, 4: 4, 5: 5}
    atc_nodes["parent_code"] = [
        None if l == 1 else c[:plen[l]] for c, l in zip(atc_nodes.atc_code, atc_nodes.level)
    ]
    RAW["atc_class"] = atc_nodes

    # --- Approval ---
    RAW["approval"] = keep(
        q("SELECT * FROM approval"),
        ["struct_id", "approval", "type", "applicant", "orphan"],
        "approval",
    )

    # --- FAERS (sinyal signifikan llr > llr_threshold, maks 20 teratas per obat) ---
    fa = q("SELECT * FROM faers")
    n_faers = len(fa)
    if {"llr", "llr_threshold"} <= set(fa.columns):
        fa = fa[fa.llr > fa.llr_threshold]
    fa = (
        fa.sort_values(["llr", "meddra_code"], ascending=[False, True], kind="mergesort")
        .drop_duplicates(["struct_id", "meddra_code"])
        .sort_values("struct_id", kind="mergesort")
        .groupby("struct_id")
        .head(20)
    )
    RAW["adverse_signal"] = keep(fa, ["struct_id", "meddra_code", "meddra_name", "llr"], "adverse_signal")

    PROV["drugcentral"] = {
        "akses": "PostgreSQL lokal (pg_restore)" if USE_LOCAL_PG else "instance PostgreSQL publik",
        "host": f"{DC_CONN['host']}:{DC_CONN['port']}",
        "database": DC_CONN["dbname"],
        "postgres_version": q("SELECT version()").iloc[0, 0],
        "baris_tabel_sumber": {
            "act_table_full": len(act),
            "structures": len(RAW["drug"]),
            "omop_relationship": len(ind),
            "atc": len(atc),
            "struct2atc": len(s2a),
            "approval": len(RAW["approval"]),
            "faers": n_faers,
        },
    }
    conn.close()
    for k in [
        "bioactivity", "target", "target_protein", "drug", "indication",
        "drug_atc", "atc_class", "approval", "adverse_signal",
    ]:
        print(f"RAW[{k!r}]: {RAW[k].shape}")


if not USE_SNAPSHOT:
    extract_drugcentral()

port 5433 (unmtid-dbs.net): TERBUKA
target dengan atribut tidak konsisten antar-baris: 0
RAW['bioactivity']: (22364, 11)
RAW['target']: (3273, 5)
RAW['target_protein']: (4021, 2)
RAW['drug']: (5116, 7)
RAW['indication']: (42515, 6)
RAW['drug_atc']: (5430, 2)
RAW['atc_class']: (6875, 4)
RAW['approval']: (4128, 5)
RAW['adverse_signal']: (32188, 4)


### 1B. Ekstraksi UniProt
Mengambil metadata protein, organisme, family protein, dan anotasi Gene Ontology (GO) berdasarkan `accession` target DrugCentral. Spasi berlebih pada `go_name` dan `family` dibersihkan dengan `.str.strip()`.

In [ ]:
UPROTKB_FIELDS = [
    "accession", "id", "gene_primary", "protein_name", "organism_name",
    "organism_id", "length", "protein_families",
    "cc_subcellular_location", "go_p", "go_f", "go_c",
]


def extract_uniprot():
    accs = sorted(RAW["target_protein"].accession.dropna().unique())
    parts, hdrs = [], {}
    for i in range(0, len(accs), 40):
        query = " OR ".join(f"accession:{a}" for a in accs[i : i + 40])
        resp = requests.get(
            "https://rest.uniprot.org/uniprotkb/search",
            params={"query": query, "fields": ",".join(UPROTKB_FIELDS), "format": "tsv", "size": 500},
            timeout=60,
        )
        resp.raise_for_status()
        parts.append(pd.read_csv(io.StringIO(resp.text), sep="\t"))
        hdrs = {k: v for k, v in resp.headers.items() if k.lower().startswith("x-uniprot")}
        time.sleep(0.3)

    up = pd.concat(parts, ignore_index=True)
    assert up.shape[1] == len(UPROTKB_FIELDS), up.columns.tolist()
    up.columns = UPROTKB_FIELDS
    up = up.drop_duplicates("accession")
    not_found = sorted(set(accs) - set(up.accession))
    print(up.shape, "| accession tidak ditemukan:", len(not_found))

    RAW["protein"] = up[["accession", "id", "gene_primary", "protein_name", "organism_id", "length"]]
    RAW["organism"] = up[["organism_id", "organism_name"]].drop_duplicates()

    def explode_terms(col, pattern, names):
        rows = []
        for acc, txt in zip(up.accession, up[col]):
            if isinstance(txt, str):
                for m in re.finditer(pattern, txt):
                    rows.append((acc, *[g.strip() for g in m.groups()]))
        return pd.DataFrame(rows, columns=["accession", *names]).drop_duplicates()

    go_pat = r"([^;\[]+?)\s*\[(GO:\d{7})\]"
    go = []
    for col, kind in [
        ("go_p", "BiologicalProcess"),
        ("go_f", "MolecularFunction"),
        ("go_c", "CellularComponent"),
    ]:
        t = explode_terms(col, go_pat, ["go_name", "go_id"])
        t["go_kind"] = kind
        go.append(t)
    go = pd.concat(go, ignore_index=True)
    RAW["protein_go"] = go[["accession", "go_id"]].drop_duplicates()
    RAW["go_term"] = go[["go_id", "go_name", "go_kind"]].drop_duplicates("go_id")

    fam = up[["accession", "protein_families"]].dropna()
    fam = fam.assign(family=fam.protein_families.str.split(r";\s*")).explode("family")
    fam["family"] = fam["family"].str.strip()
    RAW["protein_family"] = fam[fam["family"] != ""][["accession", "family"]].drop_duplicates()

    PROV["uniprot"] = {
        "endpoint": "https://rest.uniprot.org/uniprotkb/search",
        "kueri": "accession:<ID> OR ... (40 accession per permintaan, format=tsv)",
        "fields": UPROTKB_FIELDS,
        "header_respons_x_uniprot": hdrs,
        "accession_diminta": len(accs),
        "accession_tidak_ditemukan": len(not_found),
    }

if not USE_SNAPSHOT:
    extract_uniprot()

(3426, 12) | accession tidak ditemukan: 0


### 1C. Ekstraksi Reactome
Mengunduh daftar pathway *Homo sapiens*, hierarki induk-anaknya, serta pemetaan `UniProt -> Reactome`.

In [ ]:
def extract_reactome():
    accs = set(RAW["target_protein"].accession.dropna())
    base = "https://reactome.org/download/current/"
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    }
    reactome_files = {}

    def get_tsv(name, cols):
        r = requests.get(base + name, headers=HEADERS, timeout=120)
        print(name, "->", r.status_code, f"{len(r.content)/1e6:.1f} MB")
        r.raise_for_status()
        reactome_files[name] = {
            "bytes": len(r.content),
            "last_modified": r.headers.get("Last-Modified"),
            "etag": r.headers.get("ETag"),
        }
        return pd.read_csv(io.BytesIO(r.content), sep="\t", header=None, names=cols)

    pw = get_tsv("ReactomePathways.txt", ["pathway_id", "pathway_name", "species"])
    rel = get_tsv("ReactomePathwaysRelation.txt", ["parent_id", "child_id"])
    u2r = get_tsv(
        "UniProt2Reactome.txt",
        ["accession", "pathway_id", "url", "pathway_name", "evidence", "species"],
    )

    pw = pw[pw.species == "Homo sapiens"]
    rel = rel[rel.parent_id.isin(pw.pathway_id) & rel.child_id.isin(pw.pathway_id)]
    u2r = u2r[(u2r.species == "Homo sapiens") & u2r.accession.isin(accs)]

    RAW["pathway"] = pw[["pathway_id", "pathway_name"]]
    RAW["pathway_hierarchy"] = rel
    RAW["protein_pathway"] = u2r[["accession", "pathway_id"]].drop_duplicates()
    PROV["reactome"] = {
        "base_url": base,
        "berkas": reactome_files,
        "filter": "species == Homo sapiens; protein dibatasi ke accession target obat",
    }
    print({k: len(v) for k, v in RAW.items()})


if not USE_SNAPSHOT:
    extract_reactome()

ReactomePathways.txt -> 200 1.6 MB
ReactomePathwaysRelation.txt -> 200 0.6 MB
UniProt2Reactome.txt -> 200 43.2 MB
{'bioactivity': 22364, 'target': 3273, 'target_protein': 4021, 'drug': 5116, 'indication': 42515, 'drug_atc': 5430, 'atc_class': 6875, 'approval': 4128, 'adverse_signal': 32188, 'protein': 3426, 'organism': 249, 'protein_go': 74573, 'go_term': 10106, 'protein_family': 3216, 'pathway': 2889, 'pathway_hierarchy': 2905, 'protein_pathway': 9341}


### 1D. Simpan Snapshot Data Mentah
Menyimpan hasil ekstraksi ke `raw_cache/raw_snapshot.pkl` (atau Google Drive) agar pembersihan, validasi, dan ekspor dapat diulang tanpa menghubungi ketiga sumber lagi.

In [ ]:
if not USE_SNAPSHOT:
    SNAP.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle({"RAW": RAW, "PROV": PROV}, SNAP)
    print(f"Snapshot data mentah disimpan: {SNAP} ({SNAP.stat().st_size / 1e6:.1f} MB)")
else:
    print("Menggunakan data mentah yang sudah ada; tidak ada snapshot baru.")

missing = [k for k in RAW_KEYS if k not in RAW]
assert not missing, f"RAW belum lengkap: {missing}"

Snapshot data mentah disimpan: /content/drive/MyDrive/kgqa/raw_snapshot.pkl (13.2 MB)


## 2. Skema Dataset Final (20 Tabel Relasional, Netral Ontologi)
Satu sumber kebenaran untuk urutan/tipe kolom, kunci utama (*Primary Key*), relasi antar-tabel (*Foreign Key*), validasi, `manifest.json`, dan sheet `README`.

In [ ]:
def _t(description, key, cols):
    return {
        "description": description,
        "key": key,
        "columns": [
            dict(name=c[0], type=c[1], description=c[2], ref=(c[3] if len(c) > 3 else None))
            for c in cols
        ],
    }


SCHEMA = {
    "organism": _t("Organisme (spesies) sumber protein.", ["organism_id"], [
        ("organism_id", "int", "ID taksonomi NCBI (dari UniProt)."),
        ("organism_name", "string", "Nama organisme menurut UniProt. Bila satu ID muncul dengan >1 nama, dipakai nama terpendek."),
    ]),
    "protein": _t("Entri protein UniProt yang menjadi komponen target obat. Entri berstatus 'deleted' dibuang.", ["accession"], [
        ("accession", "string", "Accession UniProtKB."),
        ("id", "string", "Entry name UniProtKB."),
        ("gene_primary", "string", "Simbol gen utama (case-sensitive); kosong untuk sebagian protein."),
        ("protein_name", "string", "Nama protein."),
        ("organism_id", "int", "Organisme protein.", "organism.organism_id"),
        ("length", "int", "Panjang sekuens (asam amino)."),
    ]),
    "protein_family": _t("Family protein menurut UniProt (string protein_families dipisah berdasarkan titik-koma ';').", ["accession", "family"], [
        ("accession", "string", "Protein.", "protein.accession"),
        ("family", "string", "Nama family (satu segmen utuh dari string protein_families UniProt)."),
    ]),
    "target": _t("Target obat menurut DrugCentral: bisa protein tunggal, kompleks, atau keluarga protein.", ["target_id"], [
        ("target_id", "int", "ID target internal DrugCentral."),
        ("target_name", "string", "Nama target."),
        ("target_class", "string", "Kelas fungsional target (mis. Enzyme, Kinase, GPCR)."),
        ("organism", "string", "Nama organisme target menurut DrugCentral (teks bebas, bukan FK)."),
        ("tdl", "string", "Target Development Level dari DrugCentral (Tclin, Tchem, Tbio, Tdark)."),
    ]),
    "target_protein": _t("Relasi target -> protein. Satu target dapat memiliki beberapa protein (kompleks/keluarga).", ["target_id", "accession"], [
        ("target_id", "int", "Target.", "target.target_id"),
        ("accession", "string", "Protein komponen.", "protein.accession"),
    ]),
    "bioactivity": _t("Satu baris tabel act_table_full DrugCentral = satu pengukuran/anotasi aktivitas obat terhadap target.", ["act_id"], [
        ("act_id", "string", "ID sintetis ACT#######, deterministik."),
        ("struct_id", "int", "Obat.", "drug.struct_id"),
        ("target_id", "int", "Target.", "target.target_id"),
        ("act_value", "float", "Nilai aktivitas; kosong pada sebagian baris."),
        ("act_unit", "string", "Satuan nilai aktivitas (kosong pada ekspor sumber saat ini)."),
        ("act_type", "string", "Jenis ukuran (Ki, IC50, Kd, EC50, dst.)."),
        ("act_comment", "string", "Komentar eksperimen dari sumber."),
        ("act_source", "string", "Sumber data aktivitas menurut DrugCentral."),
        ("relation", "string", "Operator relasi nilai dari sumber (=, <, >, ~)."),
        ("moa", "float", "1.0 bila baris ini adalah anotasi mekanisme aksi (MoA); selain itu kosong."),
        ("action_type", "string", "Jenis aksi (INHIBITOR, AGONIST, dst.); kosong pada mayoritas baris."),
    ]),
    "drug": _t("Zat aktif obat (tabel structures DrugCentral).", ["struct_id"], [
        ("struct_id", "int", "ID struktur internal DrugCentral; dipakai sebagai ID obat."),
        ("name", "string", "Nama obat."),
        ("cas_reg_no", "string", "Nomor registri CAS."),
        ("smiles", "string", "Struktur dalam notasi SMILES."),
        ("inchikey", "string", "InChIKey."),
        ("cd_molweight", "float", "Berat molekul."),
        ("cd_formula", "string", "Rumus molekul."),
        ("first_introduced", "date", "Dari baris approval bertipe 'YEAR INTRODUCED' (YYYY-MM-DD); kosong bila tidak ada."),
    ]),
    "association": _t("Relasi obat-penyakit dari tabel omop_relationship DrugCentral (indikasi, kontraindikasi, off-label, dll.).", ["assoc_id"], [
        ("assoc_id", "string", "ID sintetis ASSOC######, deterministik."),
        ("struct_id", "int", "Obat.", "drug.struct_id"),
        ("concept_id", "int", "Penyakit.", "disease.concept_id"),
        ("relationship_name", "string", "Jenis hubungan menurut sumber (indication, contraindication, off-label use, dst.)."),
    ]),
    "disease": _t("Konsep penyakit/kondisi dari tabel omop_relationship.", ["concept_id"], [
        ("concept_id", "int", "ID konsep dari omop_relationship DrugCentral."),
        ("concept_name", "string", "Label utama = nama yang paling sering muncul untuk concept_id tersebut."),
        ("snomed_conceptid", "int", "ID SNOMED CT (modus bila bervariasi antar-baris). Gunakan JSON untuk presisi penuh >15 digit."),
        ("umls_cui", "string", "UMLS CUI (modus bila bervariasi antar-baris)."),
    ]),
    "disease_synonym": _t("Label alternatif penyakit yang dipetakan ke concept_id yang sama oleh sumber.", ["concept_id", "alt_label"], [
        ("concept_id", "int", "Penyakit.", "disease.concept_id"),
        ("alt_label", "string", "Label alternatif."),
    ]),
    "atc_class": _t("Simpul hierarki ATC (Anatomical Therapeutic Chemical), level 1-5.", ["atc_code"], [
        ("atc_code", "string", "Kode ATC."),
        ("atc_name", "string", "Nama kelas/zat."),
        ("level", "int", "Level hierarki 1-5."),
        ("parent_code", "string", "Kode induk (diturunkan dari awalan kode; kosong pada level 1).", "atc_class.atc_code"),
    ]),
    "drug_atc": _t("Relasi obat -> kelas ATC.", ["struct_id", "atc_code"], [
        ("struct_id", "int", "Obat.", "drug.struct_id"),
        ("atc_code", "string", "Kode ATC.", "atc_class.atc_code"),
    ]),
    "approval": _t("Persetujuan regulator terhadap obat. Baris 'YEAR INTRODUCED' dipindah ke drug.first_introduced.", ["approval_id"], [
        ("approval_id", "string", "ID sintetis APR######, deterministik."),
        ("struct_id", "int", "Obat.", "drug.struct_id"),
        ("agency", "string", "Nama lembaga regulator (FDA, EMA, PMDA, dst.)."),
        ("approval_date", "date", "Tanggal persetujuan (YYYY-MM-DD); kosong pada sebagian baris."),
        ("applicant", "string", "Pemohon/perusahaan; kosong pada sebagian baris."),
    ]),
    "adverse_signal": _t("Sinyal statistik obat-kejadian merugikan dari FAERS (llr > llr_threshold, maks 20 teratas per obat).", ["signal_id"], [
        ("signal_id", "string", "ID sintetis SIG######, deterministik."),
        ("struct_id", "int", "Obat.", "drug.struct_id"),
        ("meddra_code", "int", "Kejadian merugikan.", "adverse_event.meddra_code"),
        ("llr", "float", "Log-likelihood ratio sinyal; makin besar makin kuat."),
    ]),
    "adverse_event": _t("Istilah kejadian merugikan standar MedDRA.", ["meddra_code"], [
        ("meddra_code", "int", "Kode MedDRA."),
        ("meddra_name", "string", "Istilah MedDRA."),
    ]),
    "pathway": _t("Pathway Reactome (hanya Homo sapiens).", ["pathway_id"], [
        ("pathway_id", "string", "ID stabil Reactome."),
        ("pathway_name", "string", "Nama pathway."),
    ]),
    "pathway_hierarchy": _t("Hubungan induk-anak antar-pathway (DAG; satu pathway dapat memiliki >1 induk).", ["parent_id", "child_id"], [
        ("parent_id", "string", "Pathway induk.", "pathway.pathway_id"),
        ("child_id", "string", "Pathway anak.", "pathway.pathway_id"),
    ]),
    "protein_pathway": _t("Relasi protein -> pathway (berkas UniProt2Reactome.txt).", ["accession", "pathway_id"], [
        ("accession", "string", "Protein.", "protein.accession"),
        ("pathway_id", "string", "Pathway.", "pathway.pathway_id"),
    ]),
    "go_term": _t("Istilah Gene Ontology yang menganotasi protein target.", ["go_id"], [
        ("go_id", "string", "ID GO (GO:#######)."),
        ("go_name", "string", "Nama istilah GO."),
        ("go_kind", "string", "BiologicalProcess | MolecularFunction | CellularComponent."),
    ]),
    "protein_go": _t("Relasi protein -> istilah GO.", ["accession", "go_id"], [
        ("accession", "string", "Protein.", "protein.accession"),
        ("go_id", "string", "Istilah GO.", "go_term.go_id"),
    ]),
}

print(f"{len(SCHEMA)} tabel, {sum(len(s['columns']) for s in SCHEMA.values())} kolom terdefinisi")

20 tabel, 74 kolom terdefinisi


## 3. Pembersihan dan Normalisasi
Keputusan pembersihan data (murni relasional, tanpa pemodelan ontologi):
1. Entri UniProt berstatus `deleted` dibuang beserta relasinya (`target_protein`, `protein_go`, `protein_pathway`, `protein_family`).
2. Satu nama per `organism_id` (diambil nama terpendek).
3. Penyakit: nama paling sering menjadi `concept_name`; variasi lainnya dipisahkan ke `disease_synonym`. SNOMED/UMLS yang bervariasi diambil modusnya.
4. `approval` bertipe `YEAR INTRODUCED` dipindah ke `drug.first_introduced`.
5. Sinyal FAERS dibatasi pada `llr > llr_threshold`, maksimal 20 teratas per obat.
6. ID sintetis (`ACT`, `ASSOC`, `APR`, `SIG`) deterministik berdasarkan urutan isi baris.
7. Whitespace di awal/akhir `go_term.go_name` dan `protein_family.family` dibersihkan agar rapi bahkan jika memuat snapshot lama.

In [ ]:
def build_tables(raw):
    T = {}

    # ---- Organism: satu nama per ID (nama terpendek) ----
    org = raw["organism"].dropna(subset=["organism_id"]).copy()
    org["organism_id"] = num(org.organism_id)
    org = (
        org.assign(_n=org.organism_name.str.len())
        .sort_values(["_n", "organism_name"], kind="mergesort")
        .drop(columns="_n")
    )
    T["organism"] = org.drop_duplicates("organism_id")

    # ---- Protein: buang entri UniProt 'deleted' ----
    p = raw["protein"].copy()
    p["organism_id"] = num(p.organism_id)
    p["length"] = num(p.length)
    bad = p[(p.protein_name == "deleted") | p.organism_id.isna()]
    print(f"protein 'deleted' dibuang: {len(bad)} | sisa: {len(p) - len(bad)}")
    T["protein"] = p.drop(index=bad.index).copy()

    # ---- Family protein: bersihkan whitespace & teruskan apa adanya ----
    pf = raw["protein_family"].dropna(subset=["family"]).copy()
    pf["family"] = pf["family"].astype(str).str.strip()
    T["protein_family"] = pf[pf["family"] != ""].drop_duplicates(["accession", "family"])

    # ---- Target ----
    t = raw["target"].copy()
    t["target_id"] = num(t.target_id)
    T["target"] = t

    # ---- Bioactivity ----
    b = raw["bioactivity"].copy()
    b["struct_id"] = num(b.struct_id)
    b["target_id"] = num(b.target_id)
    b["act_value"] = pd.to_numeric(b.act_value, errors="coerce")
    T["bioactivity"] = b

    # ---- Drug-disease association & Disease ----
    i = raw["indication"].copy()
    for c in ["struct_id", "concept_id", "snomed_conceptid"]:
        i[c] = num(i[c])
    i = i.dropna(subset=["concept_id"]).copy()
    T["association"] = assign_ids(
        i[["struct_id", "concept_id", "relationship_name"]],
        "assoc_id",
        "ASSOC",
        6,
    )

    cnt = (
        i.dropna(subset=["concept_name"])
        .groupby(["concept_id", "concept_name"])
        .size()
        .reset_index(name="n")
        .sort_values(["concept_id", "n", "concept_name"], ascending=[True, False, True], kind="mergesort")
    )
    canon = cnt.drop_duplicates("concept_id")[["concept_id", "concept_name"]]
    alt = (
        cnt.merge(canon, on=["concept_id", "concept_name"], how="left", indicator=True)
        .query("_merge == 'left_only'")[["concept_id", "concept_name"]]
        .rename(columns={"concept_name": "alt_label"})
    )

    def mode(s):
        return s.dropna().mode().iloc[0] if s.notna().any() else pd.NA

    attrs = (
        i.groupby("concept_id")
        .agg(snomed_conceptid=("snomed_conceptid", mode), umls_cui=("umls_cui", mode))
        .reset_index()
    )
    attrs["snomed_conceptid"] = num(attrs.snomed_conceptid)

    ids = pd.DataFrame({"concept_id": sorted(i.concept_id.dropna().unique())})
    ids["concept_id"] = num(ids.concept_id)
    T["disease"] = ids.merge(canon, on="concept_id", how="left").merge(attrs, on="concept_id", how="left")
    T["disease_synonym"] = alt
    print("penyakit dengan >1 nama:", len(alt.concept_id.unique()), "| label alternatif:", len(alt))

    # ---- ATC ----
    a = raw["atc_class"].copy()
    a["level"] = num(a.level)
    T["atc_class"] = a
    T["drug_atc"] = raw["drug_atc"].copy()

    # ---- Approval & Drug ----
    ap_raw = raw["approval"].copy()
    ap_raw["struct_id"] = num(ap_raw.struct_id)
    ap_raw["approval_date"] = pd.to_datetime(ap_raw.approval, errors="coerce").dt.strftime("%Y-%m-%d")
    print(f"tanggal approval kosong: {int(ap_raw.approval_date.isna().sum())} / {len(ap_raw)}")
    intro = (
        ap_raw[ap_raw.type == "YEAR INTRODUCED"][["struct_id", "approval_date"]]
        .rename(columns={"approval_date": "first_introduced"})
        .drop_duplicates("struct_id")
    )
    ap = ap_raw[(ap_raw.type != "YEAR INTRODUCED") & ap_raw.type.notna()].copy()
    ap = ap.rename(columns={"type": "agency"})
    T["approval"] = assign_ids(ap[["struct_id", "agency", "approval_date", "applicant"]], "approval_id", "APR", 6)

    d = raw["drug"].copy()
    d["struct_id"] = num(d.struct_id)
    T["drug"] = d.merge(intro, on="struct_id", how="left")

    # ---- Adverse events ----
    s = raw["adverse_signal"].copy()
    s["struct_id"] = num(s.struct_id)
    s["meddra_code"] = num(s.meddra_code)
    s["llr"] = pd.to_numeric(s.llr, errors="coerce")
    print("meddra_code dengan >1 nama:", int((s.groupby("meddra_code").meddra_name.nunique() > 1).sum()))
    T["adverse_signal"] = assign_ids(s[["struct_id", "meddra_code", "llr"]], "signal_id", "SIG", 6)
    T["adverse_event"] = s.drop_duplicates("meddra_code")[["meddra_code", "meddra_name"]]

    # ---- GO Term (bersihkan spasi awal/akhir go_name) ----
    gt = raw["go_term"].copy()
    gt["go_name"] = gt["go_name"].where(gt["go_name"].isna(), gt["go_name"].astype(str).str.strip())
    T["go_term"] = gt.drop_duplicates("go_id")

    # ---- Tabel yang diteruskan apa adanya ----
    for n in ["target_protein", "protein_go", "pathway", "pathway_hierarchy", "protein_pathway"]:
        T[n] = raw[n].copy()

    # ---- Buang relasi yang menunjuk ke protein yang dibuang ----
    valid = set(T["protein"].accession)
    for n in ["target_protein", "protein_go", "protein_pathway", "protein_family"]:
        before = len(T[n])
        T[n] = T[n][T[n].accession.isin(valid)]
        print(f"{n:22s}{before:>8} -> {len(T[n]):>8}  (dibuang {before - len(T[n])})")
    print(
        "target tanpa komponen protein:",
        int((~T["target"].target_id.isin(T["target_protein"].target_id)).sum()),
        "/",
        len(T["target"]),
    )
    return finalize(T)


def finalize(T):
    out = {}
    for name, spec in SCHEMA.items():
        df = T[name]
        names = [c["name"] for c in spec["columns"]]
        missing = [n for n in names if n not in df.columns]
        assert not missing, f"{name}: kolom hilang {missing}"
        df = df[names].copy()
        for c in spec["columns"]:
            n, ty = c["name"], c["type"]
            if ty == "int":
                df[n] = num(df[n])
            elif ty == "float":
                df[n] = pd.to_numeric(df[n], errors="coerce").astype("float64")
            else:
                df[n] = df[n].astype(object).where(df[n].notna(), None)
        out[name] = df.sort_values(spec["key"], kind="mergesort").reset_index(drop=True)
    return out


TABLES = build_tables(RAW)
print("\nRingkasan baris tiap tabel:")
for k, v in TABLES.items():
    print(f"  {k:22s}: {len(v):>7,}")

protein 'deleted' dibuang: 89 | sisa: 3337
penyakit dengan >1 nama: 118 | label alternatif: 138
tanggal approval kosong: 531 / 4128
meddra_code dengan >1 nama: 0
target_protein            4021 ->     3931  (dibuang 90)
protein_go               74573 ->    74573  (dibuang 0)
protein_pathway           9341 ->     9340  (dibuang 1)
protein_family            3216 ->     3216  (dibuang 0)
target tanpa komponen protein: 80 / 3273

Ringkasan baris tiap tabel:
  organism              :     246
  protein               :   3,337
  protein_family        :   3,216
  target                :   3,273
  target_protein        :   3,931
  bioactivity           :  22,364
  drug                  :   5,116
  association           :  42,515
  disease               :   3,963
  disease_synonym       :     138
  atc_class             :   6,875
  drug_atc              :   5,430
  approval              :   3,971
  adverse_signal        :  32,188
  adverse_event         :   4,208
  pathway               :   2,889

## 4. Validasi Integritas Data
Memeriksa keunikan seluruh *Primary Key*, integritas referensial seluruh *Foreign Key*, kebersihan *whitespace* pada `go_name`, dan diagnostik skala nilai `act_value`. Proses berhenti dengan `AssertionError` jika ada pelanggaran.

In [ ]:
def validate(T):
    problems = []
    print("== Kunci unik (Primary Key) ==")
    for name, spec in SCHEMA.items():
        dup = int(T[name].duplicated(spec["key"]).sum())
        nul = int(T[name][spec["key"]].isna().any(axis=1).sum())
        ok = dup == 0 and nul == 0
        print(f"{'OK ' if ok else 'XXX'} {name:22s} kunci={spec['key']}  duplikat={dup}  kosong={nul}")
        if not ok:
            problems.append(f"kunci {name}")

    print("\n== Integritas referensial (Foreign Key) ==")
    for name, spec in SCHEMA.items():
        for c in spec["columns"]:
            if c["ref"]:
                pt, pc = c["ref"].split(".")
                vals = T[name][c["name"]].dropna()
                orphan = int((~vals.isin(T[pt][pc])).sum())
                print(f"{'OK ' if orphan == 0 else 'XXX'} {name}.{c['name']:18s} -> {c['ref']:30s} yatim={orphan}")
                if orphan:
                    problems.append(f"FK {name}.{c['name']}")

    print("\n== Pemeriksaan kualitas tambahan ==")
    bad_go = int(T["go_term"]["go_name"].dropna().map(lambda s: s != s.strip()).sum())
    print(f"{'OK ' if bad_go == 0 else 'XXX'} go_term.go_name dengan spasi awal/akhir: {bad_go}")
    if bad_go:
        problems.append("whitespace go_term.go_name")

    fake_nan_pf = int((T["protein_family"]["family"] == "nan").sum())
    print(f"{'OK ' if fake_nan_pf == 0 else 'XXX'} protein_family dengan nilai 'nan' string: {fake_nan_pf}")
    if fake_nan_pf:
        problems.append("protein_family 'nan'")

    fake_nan_go = int((T["go_term"]["go_name"] == "nan").sum())
    print(f"{'OK ' if fake_nan_go == 0 else 'XXX'} go_term.go_name dengan nilai 'nan' string: {fake_nan_go}")
    if fake_nan_go:
        problems.append("go_term 'nan'")


    print("\n== Diagnostik nilai bioaktivitas (untuk dokumentasi skala) ==")
    diag = (
        T["bioactivity"]
        .dropna(subset=["act_value"])
        .groupby("act_type")["act_value"]
        .agg(["count", "min", "median", "max"])
    )
    print(diag.to_string())
    print("\n   (median di kisaran 6-9 berarti nilai sudah skala -log10 M;")
    print("    median ratusan/jutaan berarti satuan molar/nM mentah)")

    n_entity = sum(
        len(T[n])
        for n in [
            "organism", "protein", "target", "drug", "disease",
            "atc_class", "approval", "adverse_signal", "adverse_event",
            "pathway", "go_term",
        ]
    )
    print(f"\n   baris pada tabel entitas: {n_entity:,} (syarat tugas: >= 100)")

    assert not problems, f"Validasi gagal: {problems}"
    print("\nSEMUA PEMERIKSAAN LOLOS")


validate(TABLES)

== Kunci unik (Primary Key) ==
OK  organism               kunci=['organism_id']  duplikat=0  kosong=0
OK  protein                kunci=['accession']  duplikat=0  kosong=0
OK  protein_family         kunci=['accession', 'family']  duplikat=0  kosong=0
OK  target                 kunci=['target_id']  duplikat=0  kosong=0
OK  target_protein         kunci=['target_id', 'accession']  duplikat=0  kosong=0
OK  bioactivity            kunci=['act_id']  duplikat=0  kosong=0
OK  drug                   kunci=['struct_id']  duplikat=0  kosong=0
OK  association            kunci=['assoc_id']  duplikat=0  kosong=0
OK  disease                kunci=['concept_id']  duplikat=0  kosong=0
OK  disease_synonym        kunci=['concept_id', 'alt_label']  duplikat=0  kosong=0
OK  atc_class              kunci=['atc_code']  duplikat=0  kosong=0
OK  drug_atc               kunci=['struct_id', 'atc_code']  duplikat=0  kosong=0
OK  approval               kunci=['approval_id']  duplikat=0  kosong=0
OK  adverse_signal     

## 5. Ekspor Dataset Final
Menghasilkan:
- `dataset/json/<tabel>.json` (20 berkas JSON UTF-8, sumber utama untuk *mapping* RML/RDF)
- `dataset/dataset.xlsx` (sheet `README` + 20 sheet tabel)
- `dataset/manifest.json` & `dataset/provenance.json`
- `dataset_export.zip` (diunduh otomatis jika dijalankan di Google Colab)

In [ ]:
def export_json(name, df):
    recs = df.astype(object).where(df.notna(), None).to_dict("records")
    with open(OUT_JSON / f"{name}.json", "w", encoding="utf-8") as f:
        f.write(
            "[\n"
            + ",\n".join(json.dumps(r, ensure_ascii=False, default=_py) for r in recs)
            + "\n]\n"
        )


manifest = {"generated_at_utc": PROV["generated_at_utc"], "tables": {}}
for name, df in TABLES.items():
    export_json(name, df)
    p = OUT_JSON / f"{name}.json"
    manifest["tables"][name] = {
        "rows": len(df),
        "key": SCHEMA[name]["key"],
        "description": SCHEMA[name]["description"],
        "columns": SCHEMA[name]["columns"],
        "json": f"json/{name}.json",
        "json_bytes": p.stat().st_size,
        "json_sha256": sha256(p),
    }

XLSX = OUT / "dataset.xlsx"
opts = {"strings_to_formulas": False, "strings_to_urls": False}
with pd.ExcelWriter(XLSX, engine="xlsxwriter", engine_kwargs={"options": opts}) as xw:
    book = xw.book
    book.formats[0].set_font_name("Arial")
    book.formats[0].set_font_size(10)

    readme = pd.DataFrame([
        {
            "tabel": n,
            "baris": m["rows"],
            "kunci": ", ".join(m["key"]),
            "deskripsi": m["description"],
        }
        for n, m in manifest["tables"].items()
    ])
    readme.to_excel(xw, sheet_name="README", index=False)
    ws = xw.sheets["README"]
    ws.freeze_panes(1, 0)
    for j, w in enumerate([24, 10, 28, 110]):
        ws.set_column(j, j, w)

    for name, df in TABLES.items():
        assert len(df) + 1 <= EXCEL_MAX, f"{name} melebihi batas baris Excel"
        df.to_excel(xw, sheet_name=name, index=False)
        ws = xw.sheets[name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)
        for j, col in enumerate(df.columns):
            sample = df[col].head(2000).astype(str).str.len()
            ws.set_column(
                j,
                j,
                int(min(60, max(len(col), sample.max() if len(sample) else 0) + 2)),
            )

manifest["xlsx"] = {
    "file": "dataset.xlsx",
    "bytes": XLSX.stat().st_size,
    "sheets": 1 + len(TABLES),
    "sha256": sha256(XLSX),
}

(OUT / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=1, default=_py), encoding="utf-8"
)
(OUT / "provenance.json").write_text(
    json.dumps(PROV, ensure_ascii=False, indent=1, default=str), encoding="utf-8"
)

summary = pd.DataFrame([
    {"tabel": n, "baris": m["rows"], "json_MB": round(m["json_bytes"] / 1e6, 2)}
    for n, m in manifest["tables"].items()
])
print(summary.to_string(index=False))
tot_json = sum(m["json_bytes"] for m in manifest["tables"].values()) / 1e6
print(f"\nTotal JSON: {tot_json:.1f} MB | dataset.xlsx: {XLSX.stat().st_size / 1e6:.1f} MB")

for f in list(OUT_JSON.glob("*.json")) + [XLSX]:
    if f.stat().st_size > 50e6:
        print(f"[PERINGATAN] {f.name} > 50 MB: GitHub memberi peringatan di atas 50 MB dan menolak di atas 100 MB.")

            tabel  baris  json_MB
         organism    246     0.02
          protein   3337     0.84
   protein_family   3216     0.28
           target   3273     0.48
   target_protein   3931     0.17
      bioactivity  22364     6.51
             drug   5116     1.31
      association  42515     4.71
          disease   3963     0.50
  disease_synonym    138     0.01
        atc_class   6875     0.65
         drug_atc   5430     0.24
         approval   3971     0.50
   adverse_signal  32188     2.82
    adverse_event   4208     0.28
          pathway   2889     0.26
pathway_hierarchy   2905     0.17
  protein_pathway   9340     0.52
          go_term  10106     1.10
       protein_go  74573     3.58

Total JSON: 25.0 MB | dataset.xlsx: 6.1 MB


In [ ]:
zpath = Path("dataset_export.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(OUT.parent))

print(f"Dibuat: {zpath} ({zpath.stat().st_size / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(str(zpath))
except ImportError:
    pass

Dibuat: dataset_export.zip (7.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>